# Posed NeuroMechFly cartoons

`add_neuromechfly` draws a fly in an arbitrary posture, built from the
[NeuroMechFly v2](https://github.com/NeLy-EPFL/flygym) biomechanical model. It
runs forward kinematics on the body segments for a set of joint angles, projects
them orthographically for the chosen view, and fills the silhouette of each
segment's projected mesh as a black-outlined patch (z-ordered by depth). The
meshes and rigging are bundled, so no `flygym`/`mujoco` is needed.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt

from flyplotlib import add_neuromechfly


def show(ax, title=None):
    """Tidy an axes after drawing: equal aspect, autoscaled, no spines."""
    ax.set_aspect("equal")
    ax.autoscale()
    ax.margins(0.05)
    ax.axis("off")
    if title:
        ax.set_title(title)


# The default (no joint_angles) is the neutral standing pose.
fig, ax = plt.subplots(figsize=(4, 3))
add_neuromechfly(view="right", ax=ax)
show(ax, "neutral pose (right)")

## Anatomical views

Pick `"dorsal"`, `"lateral"`, `"frontal"` or `"caudal"`. `rotation` rotates the
cartoon in-plane (in degrees), and `xy`/`length`/`origin` place and scale it just
like `add_fly`.

In [ ]:
views = ["left", "right", "front", "back", "top", "bottom"]
fig, axes = plt.subplots(1, len(views), figsize=(len(views) * 2.5, 3.5))
for ax, view in zip(axes, views, strict=False):
    add_neuromechfly(view=view, ax=ax)
    show(ax, view)
fig.tight_layout()

## Setting joint angles

`joint_angles` is a mapping of DOF name (`"{parent}-{child}-{axis}"`) to angle in
degrees. Any DOF you leave out keeps its neutral value, so you only specify what
you want to move. Use `neutral_pose()` to discover the available DOF names, or
`degrees=False` to feed flygym simulation output (radians) directly.

In [ ]:
extended = {
    "c_thorax-lf_coxa-pitch": -20,
    "lf_coxa-lf_trochanterfemur-pitch": -90,
    "lf_trochanterfemur-lf_tibia-pitch": 40,
}

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
add_neuromechfly(view="right", ax=axes[0], smoothing=5e-3)
add_neuromechfly(view="right", joint_angles=extended, ax=axes[1])
show(axes[0], "neutral")
show(axes[1], "left front leg extended")
fig.tight_layout()

## Hiding, recoloring and styling parts

`exclude` takes regular expressions matched against segment names (e.g. `"wing"`,
or `"r._"` for the right legs). `colors` is a `{pattern: color}` mapping; segments
otherwise use the model's own colors. `grayscale=True` desaturates everything.

Outlines are black by default — pass `edgecolor`/`linewidth` to restyle them or
`edgecolor="none"` to drop them. Silhouettes are lightly smoothed to hide the
mesh's triangle "staircase"; `smoothing` (a fraction of the body size) tunes this,
and `smoothing=0` keeps the raw outline.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.3))

add_neuromechfly(view="top", ax=axes[0])
show(axes[0], "default (black edges)")

add_neuromechfly(view="top", exclude=["wing", "haltere"], ax=axes[1])
show(axes[1], "wings & halteres hidden")

add_neuromechfly(
    view="top",
    colors={"eye": "black", "_tibia$": "#1f77b4", "_tarsus": "#1f77b4"},
    edgecolor="white",
    ax=axes[2],
)
show(axes[2], "recolored + white edges")

add_neuromechfly(view="top", edgecolor="none", grayscale=True, ax=axes[3])
show(axes[3], "no edges, grayscale")
fig.tight_layout()

## Coloring legs by a value

Because `colors` keys are regular expressions, you can map a per-leg (or
per-segment) scalar onto a colormap — handy for showing measured or simulated
quantities on the body. Here each leg is tinted by a value via a `ScalarMappable`
that also drives a colorbar.

In [ ]:
import matplotlib as mpl

leg_values = {"lf": 0.1, "lm": 0.45, "lh": 0.8, "rf": 0.25, "rm": 0.6, "rh": 0.95}
mappable = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(0, 1), cmap="viridis")
colors = {rf"^{leg}_": mappable.to_rgba(v) for leg, v in leg_values.items()}

fig, ax = plt.subplots(figsize=(5, 4))
add_neuromechfly(
    view="top", colors={"_(coxa|trochanterfemur)": "0.85", **colors}, ax=ax
)
show(ax, "legs colored by value")
fig.colorbar(mappable, ax=ax, shrink=0.6, label="value")